# Marketing Brand Cockpit — extract

Pulls the 3 long-format CSVs the dashboard reads:

| File | Grain | Source |
|---|---|---|
| `sales.csv` | day × city × L0/L1/L2 × manuf × brand | `viz.category_etls_fact_sales_order_item_details` + `dwh.dim_product` |
| `impressions.csv` | day × city × L0/L1/L2 × manuf × brand × ptype × pid × normalised page | `category_etls.pid_impressions_and_atc` + `dwh.dim_product` |
| `keywords.csv` | day × keyword × manuf × brand × ptype | `search_etls.keyword_city_product_additions` + `ads_etls.ads_*_searches` + `dwh.dim_product` |

Everything else (brand picking, city filter, window, MS calc, SOV calc, ATC rate, etc.) happens in the FastAPI app over DuckDB. **Don't add brand/manufacturer/L filters here.**

Set `START_DATE` and `END_DATE`, then run all cells.

In [3]:
START_DATE = '2026-04-01'
END_DATE   = '2026-06-30'   # bootstrap window. For daily refresh, set START=END=yesterday.

OUT_DIR = '/home/jovyan'

import os, time, csv
from datetime import date, timedelta
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pencilbox as pb

connection = pb.get_connection("[Warehouse] Trino")

def sql(query: str) -> pd.DataFrame:
    """Small queries only. Loads everything into a DataFrame."""
    t0 = time.time()
    df = pd.read_sql_query(sql=query, con=connection)
    print(f"  ok {time.time() - t0:6.1f}s  {len(df):>10,} rows")
    return df

def run_to_parquet_chunked(sql_template: str,
                           out_path: str,
                           start: str,
                           end: str,
                           batch_size: int = 50_000) -> int:
    """Run `sql_template` once per day in [start, end], appending to a single
    Parquet. Trino has a 20-min hard cap per query, and a full-period join +
    groupby across 30+ days blows past it; per-day chunks stay well inside.

    The SQL must use the literal placeholder `{DAY}` (Python .format substitution,
    not f-string) for the date bound. Example filters:
        AND at_date_ist  = DATE '{DAY}'
        AND cart_checkout_ts_ist >= DATE '{DAY}'
          AND cart_checkout_ts_ist <  DATE '{DAY}' + INTERVAL '1' DAY
    """
    t0 = time.time()
    writer = None
    schema = None
    total = 0
    cur_day = date.fromisoformat(start)
    last_day = date.fromisoformat(end)
    while cur_day <= last_day:
        sql_day = sql_template.format(DAY=cur_day.isoformat())
        print(f'  {cur_day.isoformat()}  executing…', flush=True)
        raw = connection.raw_connection()
        cur = raw.cursor()
        td0 = time.time()
        cur.execute(sql_day)
        cols = [d[0] for d in cur.description]
        day_rows = 0
        while True:
            batch = cur.fetchmany(batch_size)
            if not batch:
                break
            df = pd.DataFrame(batch, columns=cols)
            table = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                schema = table.schema
                writer = pq.ParquetWriter(out_path, schema, compression='snappy')
            else:
                table = table.cast(schema)
            writer.write_table(table)
            day_rows += len(batch)
            print(f'    {time.time()-td0:6.1f}s  {day_rows:>10,} rows', flush=True)
        cur.close(); raw.close()
        total += day_rows
        print(f"  {cur_day.isoformat()}  done {time.time()-td0:6.1f}s  "
              f"{day_rows:,} rows  (cum {total:,})", flush=True)
        cur_day += timedelta(days=1)
    if writer is not None:
        writer.close()
    size_mb = os.path.getsize(out_path) / 1024**2 if os.path.exists(out_path) else 0
    print(f"  done {out_path}: {total:,} rows, {size_mb:.1f} MB in {time.time()-t0:.0f}s")
    return total

print(f'Window: {START_DATE} → {END_DATE}')


Window: 2026-04-01 → 2026-06-30


In [4]:
# --- Diagnostic: confirm base tables have rows in the chosen window ---
# Run this once after editing START_DATE/END_DATE. If any row_count is 0, the
# heavy queries below will also return 0 — fix the table/window before continuing.
#
# Note: dwh.fact_sales_order_item_details REQUIRES a filter on
# `order_create_dt_ist` (partition column) — Trino rejects the query otherwise.

diag_sql = f"""
SELECT 'sales_order_items' AS source,
       count(*) AS row_count,
       count(DISTINCT product_id) AS distinct_pids,
       count(DISTINCT city_name)  AS distinct_cities,
       min(cart_checkout_ts_ist)  AS min_ts,
       max(cart_checkout_ts_ist)  AS max_ts
FROM dwh.fact_sales_order_item_details
WHERE order_create_dt_ist >= DATE '{START_DATE}'
  AND order_create_dt_ist <  DATE '{END_DATE}' + INTERVAL '6' DAY
  AND cart_checkout_ts_ist >= DATE '{START_DATE}'
  AND cart_checkout_ts_ist <  DATE '{END_DATE}' + INTERVAL '1' DAY
  AND procured_quantity > 0
  AND is_internal_order = FALSE
  AND order_current_status = 'DELIVERED'

UNION ALL
SELECT 'impressions',
       count(*),
       count(DISTINCT pid),
       count(DISTINCT city),
       CAST(min(at_date_ist) AS TIMESTAMP),
       CAST(max(at_date_ist) AS TIMESTAMP)
FROM category_etls.pid_impressions_and_atc
WHERE at_date_ist BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'

UNION ALL
SELECT 'keyword_atcs',
       count(*),
       count(DISTINCT product_id),
       NULL,
       CAST(min(at_date_ist) AS TIMESTAMP),
       CAST(max(at_date_ist) AS TIMESTAMP)
FROM search_etls.keyword_city_product_additions
WHERE at_date_ist >= DATE '{START_DATE}'
  AND at_date_ist <  DATE '{END_DATE}' + INTERVAL '1' DAY
"""
sql(diag_sql)

  ok  115.8s           3 rows


,source,row_count,distinct_pids,distinct_cities,min_ts,max_ts
0,keyword_atcs,432255715,118162,NaN,2026-04-01 00:00:00.000,2026-06-30 00:00:00.000
1,sales_order_items,1285852604,117318,343.0,2026-04-01 00:00:00.082,2026-06-30 23:59:59.952
2,impressions,3333579024,287687,380.0,2026-04-01 00:00:00.000,2026-06-30 00:00:00.000


## 0. Products snapshot

Single-shot `dwh.dim_product` snapshot — small, fast (no time-cap risk). The app joins
this against pid-grain impressions/keywords at query time, so any brand/L2 rollup is
computable without re-extracting.


In [5]:
PRODUCTS_SQL = """
SELECT
  product_id,
  CASE WHEN (brand_name        IS NULL OR brand_name        = '') THEN '-' ELSE brand_name        END AS brand,
  CASE WHEN (manufacturer_name IS NULL OR manufacturer_name = '') THEN '-' ELSE manufacturer_name END AS manufacturer,
  l0_category, l1_category, l2_category,
  product_type AS ptype
FROM dwh.dim_product
WHERE is_current = TRUE
  AND product_type_id NOT IN (459,9759,11778,11791,11934)
  AND l1_Category_id  NOT IN (1013,1033,1493,1601,1658,1661,1664,1740,1741,1745,1985)
GROUP BY 1,2,3,4,5,6,7
"""
print('→ products')
products = sql(PRODUCTS_SQL)
products.to_parquet(os.path.join(OUT_DIR, 'products.parquet'), index=False, compression='snappy')
print(f'  done products.parquet: {len(products):,} rows')
products.head()


→ products
  ok   38.6s     708,745 rows
  done products.parquet: 708,745 rows


,product_id,brand,manufacturer,l0_category,l1_category,l2_category,ptype
0,35359,Everyuth,-,Personal Care,Bathing,Bathing Soaps,Soap
1,536803,Go,Parag Milk Foods Pvt Ltd,Dairy & Breakfast,Cheese,Diced Cheese,Diced Cheese
2,146060,Head & Shoulders,-,Personal Care,Shampoo & Conditioner,Shampoo & Conditioner,Shampoo
3,138316,Cameleon,-,Personal Care,Cosmetics,Eye Make-Up,dummyptype
4,505985,Pep Nails,M/s Merakin,Beauty & Cosmetics,Tools & Accessories,Nail Accessories,Nail Glue


## 1. Sales / Market Share extract + Cart Penetration (QC 2.1)

Two parallel extracts: `sales` (brand-grain, now with `brand_orders` distinct
order counts) and `city_totals` (city-grain denominator for Cart Penetration).

User Penetration was dropped — `fact_sales_order_item_details` has no
customer/user identifier column. If a customer_id-like column is added
upstream later, re-add `COUNT(DISTINCT …) AS brand_customers` here + the
matching `city_customers` in the city_totals extract.

In [ ]:
# QC 2.1 (partial) — adds COUNT(DISTINCT order_id) AS brand_orders for Cart
# Penetration. User Penetration dropped: no usable customer/user column on
# dwh.fact_sales_order_item_details (probe failed 2026-06-27).

SALES_SQL_TEMPLATE = """
WITH cat AS (
  SELECT
    product_id AS pid,
    l2_category AS l2_cat,
    l1_category AS l1_cat,
    l0_category AS l0_cat,
    CASE WHEN (brand_name IS NULL OR brand_name = '') THEN '-' ELSE brand_name END AS brand,
    CASE WHEN (manufacturer_name IS NULL OR manufacturer_name = '') THEN '-' ELSE manufacturer_name END AS manufacturer,
    CASE WHEN manufacturer_name IN (
        'HOT','Basic','Hands On Trades Pvt. Ltd.','Moonstone Ventures LLP','Udit Fresh Food Pvt. Ltd.'
    ) THEN TRUE ELSE FALSE END AS is_pl
  FROM dwh.dim_product
  WHERE is_current = TRUE
    AND product_type_id NOT IN (459,9759,11778,11791,11934)
    AND l1_Category_id NOT IN (1013,1033,1493,1601,1658,1661,1664,1740,1741,1745,1985)
  GROUP BY 1,2,3,4,5,6,7
)
SELECT
  CAST(oi.cart_checkout_ts_ist AS DATE)     AS day,
  city_name                                  AS city,
  l0_cat, l1_cat, l2_cat,
  c.manufacturer, c.brand,
  SUM(procured_quantity) AS qty,
  SUM(CASE WHEN is_pl = TRUE
           THEN (unit_selling_price * procured_quantity)
           ELSE (unit_mrp * procured_quantity) END) AS mrp,
  SUM(unit_selling_price * procured_quantity)      AS gmv,
  COUNT(DISTINCT oi.order_id)              AS brand_orders
FROM dwh.fact_sales_order_item_details oi
INNER JOIN cat c ON c.pid = oi.product_id
WHERE city_name NOT IN ('Not in service area', 'Hapur', 'test1207898732')
  AND city_name NOT LIKE '%%B2B%%'
  AND procured_quantity > 0
  AND is_internal_order = FALSE
  AND order_current_status = 'DELIVERED'
  AND cart_checkout_ts_ist >= DATE '{DAY}'
  AND cart_checkout_ts_ist <  DATE '{DAY}' + INTERVAL '1' DAY
  AND order_create_dt_ist  >= DATE '{DAY}' - INTERVAL '2' DAY
  AND order_create_dt_ist  <  DATE '{DAY}' + INTERVAL '7' DAY
GROUP BY 1,2,3,4,5,6,7
"""

# Cart Pen denominator — same window/filters as sales, grouped only by
# (day, city, L0/L1/L2) so we get the total orders any brand competes against.
CITY_TOTALS_SQL_TEMPLATE = """
WITH cat AS (
  SELECT
    product_id AS pid,
    l2_category AS l2_cat,
    l1_category AS l1_cat,
    l0_category AS l0_cat
  FROM dwh.dim_product
  WHERE is_current = TRUE
    AND product_type_id NOT IN (459,9759,11778,11791,11934)
    AND l1_Category_id NOT IN (1013,1033,1493,1601,1658,1661,1664,1740,1741,1745,1985)
  GROUP BY 1,2,3,4
)
SELECT
  CAST(oi.cart_checkout_ts_ist AS DATE)  AS day,
  city_name                               AS city,
  l0_cat, l1_cat, l2_cat,
  COUNT(DISTINCT oi.order_id)             AS city_orders
FROM dwh.fact_sales_order_item_details oi
INNER JOIN cat c ON c.pid = oi.product_id
WHERE city_name NOT IN ('Not in service area', 'Hapur', 'test1207898732')
  AND city_name NOT LIKE '%%B2B%%'
  AND procured_quantity > 0
  AND is_internal_order = FALSE
  AND order_current_status = 'DELIVERED'
  AND cart_checkout_ts_ist >= DATE '{DAY}'
  AND cart_checkout_ts_ist <  DATE '{DAY}' + INTERVAL '1' DAY
  AND order_create_dt_ist  >= DATE '{DAY}' - INTERVAL '2' DAY
  AND order_create_dt_ist  <  DATE '{DAY}' + INTERVAL '7' DAY
GROUP BY 1,2,3,4,5
"""

print("→ sales (with brand_orders for Cart Penetration)")
run_to_parquet_chunked(SALES_SQL_TEMPLATE,
                       os.path.join(OUT_DIR, "sales.parquet"),
                       START_DATE, END_DATE)
print("→ city_totals (Cart Penetration denominator)")
run_to_parquet_chunked(CITY_TOTALS_SQL_TEMPLATE,
                       os.path.join(OUT_DIR, "city_totals.parquet"),
                       START_DATE, END_DATE)
pd.read_parquet(os.path.join(OUT_DIR, "sales.parquet")).head()


→ sales (with brand_orders for Cart Penetration)
  2026-04-01  executing…


## 2. Impressions extract — pid grain

Grain: `day × city × pid` (no dim_product join, no brand/L2 rollup). DuckDB joins
this against `products` at query time — Trino runtime stays well under the 20-min cap.
Loss vs. earlier draft: `page_name` is summed (not stored). Add back when we have a
path to chunk further (per-city or per-L1 sub-chunks).


In [ ]:
IMPRESSIONS_SQL_TEMPLATE = """
SELECT
  i.at_date_ist     AS day,
  i.city,
  i.pid,
  SUM(i.impressions) AS impressions,
  SUM(i.atc)         AS atc,
  SUM(i.atc_gmv)     AS atc_gmv
FROM category_etls.pid_impressions_and_atc i
WHERE i.at_date_ist = DATE '{DAY}'
  AND i.page_name != 'Author Test Layout'
GROUP BY 1, 2, 3
"""
print("→ impressions")
run_to_parquet_chunked(IMPRESSIONS_SQL_TEMPLATE,
                       os.path.join(OUT_DIR, "impressions.parquet"),
                       START_DATE, END_DATE)
pd.read_parquet(os.path.join(OUT_DIR, "impressions.parquet")).head()


## 3. Keywords extract

Source: PDF "kw × brand × ptype atc gmv & searches" — refactored to drop the brand/L/keyword filters.

Note: this can be the largest of the 3 CSVs (millions of rows for a 30-day window across all brands × all keywords). If memory pressure shows up, chunk by date in Python or push the `GROUP BY` into Trino harder.

In [ ]:
KEYWORDS_SQL_TEMPLATE = """
WITH cat AS (
  SELECT
    dp.product_id          AS pid,
    dp.l0_category         AS l0_cat,
    dp.l1_category         AS l1_cat,
    dp.l2_category         AS l2_cat,
    dp.manufacturer_name   AS manufacturer,
    dp.brand_name,
    dp.product_type        AS ptype
  FROM dwh.dim_product dp
  WHERE dp.is_current = TRUE
    AND dp.l1_Category_id NOT IN (
      SELECT DISTINCT l1_category_id FROM category_etls.freebie_categories
    )
    AND dp.product_type_id NOT IN (
      SELECT DISTINCT product_type_id FROM category_etls.freebie_categories WHERE product_type_id != 0
    )
  GROUP BY 1,2,3,4,5,6,7
),
searches AS (
  SELECT
    at_date_ist AS day,
    keywords    AS keyword,
    SUM(searches) AS searches
  FROM (
    SELECT at_date_ist, keywords, MAX(searches) AS searches
    FROM (
      SELECT * FROM ads_etls.ads_input_searches
      UNION
      SELECT * FROM ads_etls.ads_keyword_searches
    )
    GROUP BY 1,2
  ) ks
  WHERE at_date_ist = DATE '{DAY}'
  GROUP BY 1,2
)
SELECT
  kp.at_date_ist  AS day,
  kp.search_query AS keyword,
  c.manufacturer,
  c.brand_name    AS brand,
  c.ptype,
  MAX(s.searches) AS searches,
  SUM(kp.atc_qty) AS atcs,
  SUM(kp.gmv)     AS gmv
FROM search_etls.keyword_city_product_additions kp
INNER JOIN cat c ON c.pid = kp.product_id
LEFT JOIN searches s ON s.keyword = kp.search_query AND s.day = kp.at_date_ist
WHERE kp.at_date_ist = DATE '{DAY}'
GROUP BY 1,2,3,4,5
"""
print("→ keywords")
run_to_parquet_chunked(KEYWORDS_SQL_TEMPLATE,
                       os.path.join(OUT_DIR, "keywords.parquet"),
                       START_DATE, END_DATE)
pd.read_parquet(os.path.join(OUT_DIR, "keywords.parquet")).head()


## 5. Backfill — 90-day batched extract

Replaces the 7-day pass with a checkpoint-resumable weekly batched 90-day
pull. Each batch lands in `/home/jovyan/backfill/<week_start>_<week_end>/`
with sales, city_totals, impressions (pid grain) and keywords parquets.
If the kernel dies, just re-run this cell — completed batches have a
`_DONE` sentinel and get skipped.

Run cell **6** afterwards to produce the partitioned Parquet finals for
the pod.

**Time:** ~6-8 hours for 90 days on JupyterHub (impressions is the bulk).

In [ ]:
from datetime import date, timedelta
import duckdb

BACKFILL_DIR = '/home/jovyan/backfill'
os.makedirs(BACKFILL_DIR, exist_ok=True)

BACKFILL_END   = date.today() - timedelta(days=1)
BACKFILL_START = BACKFILL_END - timedelta(days=89)   # inclusive 90 days
print(f'Backfill window: {BACKFILL_START} → {BACKFILL_END}')

# --- SQL templates (kept here so the backfill cell is self-contained) -------
# QC 2.1 (partial) — sales now carries brand_orders per (day, city, L0/L1/L2,
# manufacturer, brand). city_totals is the parallel denominator. User
# Penetration is NOT in scope — no usable user-id column on the fact table.
SALES_SQL_TEMPLATE = """
WITH cat AS (
  SELECT
    product_id AS pid,
    l2_category AS l2_cat, l1_category AS l1_cat, l0_category AS l0_cat,
    CASE WHEN (brand_name IS NULL OR brand_name='') THEN '-' ELSE brand_name END AS brand,
    CASE WHEN (manufacturer_name IS NULL OR manufacturer_name='') THEN '-' ELSE manufacturer_name END AS manufacturer,
    CASE WHEN manufacturer_name IN ('HOT','Basic','Hands On Trades Pvt. Ltd.','Moonstone Ventures LLP','Udit Fresh Food Pvt. Ltd.') THEN TRUE ELSE FALSE END AS is_pl
  FROM dwh.dim_product
  WHERE is_current = TRUE
    AND product_type_id NOT IN (459,9759,11778,11791,11934)
    AND l1_Category_id  NOT IN (1013,1033,1493,1601,1658,1661,1664,1740,1741,1745,1985)
  GROUP BY 1,2,3,4,5,6,7
)
SELECT
  CAST(oi.cart_checkout_ts_ist AS DATE) AS day, city_name AS city,
  l0_cat, l1_cat, l2_cat, c.manufacturer, c.brand,
  SUM(procured_quantity) AS qty,
  SUM(CASE WHEN is_pl=TRUE THEN unit_selling_price*procured_quantity
                           ELSE unit_mrp*procured_quantity END) AS mrp,
  SUM(unit_selling_price*procured_quantity) AS gmv,
  COUNT(DISTINCT oi.order_id)      AS brand_orders
FROM dwh.fact_sales_order_item_details oi
INNER JOIN cat c ON c.pid = oi.product_id
WHERE city_name NOT IN ('Not in service area','Hapur','test1207898732')
  AND city_name NOT LIKE '%%B2B%%'
  AND procured_quantity > 0 AND is_internal_order = FALSE
  AND order_current_status = 'DELIVERED'
  AND cart_checkout_ts_ist >= DATE '{DAY}'
  AND cart_checkout_ts_ist <  DATE '{DAY}' + INTERVAL '1' DAY
  AND order_create_dt_ist  >= DATE '{DAY}' - INTERVAL '2' DAY
  AND order_create_dt_ist  <  DATE '{DAY}' + INTERVAL '7' DAY
GROUP BY 1,2,3,4,5,6,7
"""
CITY_TOTALS_SQL_TEMPLATE = """
WITH cat AS (
  SELECT product_id AS pid,
         l2_category AS l2_cat, l1_category AS l1_cat, l0_category AS l0_cat
  FROM dwh.dim_product
  WHERE is_current = TRUE
    AND product_type_id NOT IN (459,9759,11778,11791,11934)
    AND l1_Category_id  NOT IN (1013,1033,1493,1601,1658,1661,1664,1740,1741,1745,1985)
  GROUP BY 1,2,3,4
)
SELECT
  CAST(oi.cart_checkout_ts_ist AS DATE) AS day, city_name AS city,
  l0_cat, l1_cat, l2_cat,
  COUNT(DISTINCT oi.order_id)      AS city_orders
FROM dwh.fact_sales_order_item_details oi
INNER JOIN cat c ON c.pid = oi.product_id
WHERE city_name NOT IN ('Not in service area','Hapur','test1207898732')
  AND city_name NOT LIKE '%%B2B%%'
  AND procured_quantity > 0 AND is_internal_order = FALSE
  AND order_current_status = 'DELIVERED'
  AND cart_checkout_ts_ist >= DATE '{DAY}'
  AND cart_checkout_ts_ist <  DATE '{DAY}' + INTERVAL '1' DAY
  AND order_create_dt_ist  >= DATE '{DAY}' - INTERVAL '2' DAY
  AND order_create_dt_ist  <  DATE '{DAY}' + INTERVAL '7' DAY
GROUP BY 1,2,3,4,5
"""
IMPRESSIONS_SQL_TEMPLATE = """
SELECT i.at_date_ist AS day, i.city, i.pid,
       SUM(i.impressions) AS impressions,
       SUM(i.atc)         AS atc,
       SUM(i.atc_gmv)     AS atc_gmv
FROM category_etls.pid_impressions_and_atc i
WHERE i.at_date_ist = DATE '{DAY}'
  AND i.page_name != 'Author Test Layout'
GROUP BY 1,2,3
"""
KEYWORDS_SQL_TEMPLATE = """
WITH cat AS (
  SELECT dp.product_id AS pid,
         dp.manufacturer_name AS manufacturer,
         dp.brand_name, dp.product_type AS ptype
  FROM dwh.dim_product dp
  WHERE dp.is_current = TRUE
    AND dp.l1_Category_id  NOT IN (SELECT DISTINCT l1_category_id FROM category_etls.freebie_categories)
    AND dp.product_type_id NOT IN (SELECT DISTINCT product_type_id FROM category_etls.freebie_categories WHERE product_type_id != 0)
  GROUP BY 1,2,3,4
),
searches AS (
  SELECT at_date_ist AS day, keywords AS keyword, SUM(searches) AS searches
  FROM (
    SELECT at_date_ist, keywords, MAX(searches) AS searches
    FROM (SELECT * FROM ads_etls.ads_input_searches
          UNION SELECT * FROM ads_etls.ads_keyword_searches)
    GROUP BY 1,2
  ) ks
  WHERE at_date_ist = DATE '{DAY}'
  GROUP BY 1,2
)
SELECT kp.at_date_ist AS day, kp.search_query AS keyword,
       c.manufacturer, c.brand_name AS brand, c.ptype,
       MAX(s.searches) AS searches,
       SUM(kp.atc_qty) AS atcs,
       SUM(kp.gmv)     AS gmv
FROM search_etls.keyword_city_product_additions kp
INNER JOIN cat c ON c.pid = kp.product_id
LEFT  JOIN searches s ON s.keyword = kp.search_query AND s.day = kp.at_date_ist
WHERE kp.at_date_ist = DATE '{DAY}'
GROUP BY 1,2,3,4,5
"""
PRODUCTS_SQL = """
SELECT product_id,
  CASE WHEN (brand_name IS NULL OR brand_name='') THEN '-' ELSE brand_name END AS brand,
  CASE WHEN (manufacturer_name IS NULL OR manufacturer_name='') THEN '-' ELSE manufacturer_name END AS manufacturer,
  l0_category, l1_category, l2_category, product_type AS ptype
FROM dwh.dim_product
WHERE is_current = TRUE
  AND product_type_id NOT IN (459,9759,11778,11791,11934)
  AND l1_Category_id  NOT IN (1013,1033,1493,1601,1658,1661,1664,1740,1741,1745,1985)
GROUP BY 1,2,3,4,5,6,7
"""

# --- Products snapshot (once per backfill run) ------------------------------
products_path = os.path.join(BACKFILL_DIR, 'products.parquet')
if not os.path.exists(products_path):
    print('→ products (snapshot)')
    products = sql(PRODUCTS_SQL)
    products.to_parquet(products_path, index=False, compression='snappy')
    print(f'  done {products_path}: {len(products):,} rows')
else:
    print(f'  ✓ products snapshot exists, skip')

# --- Diagnostic gate: base tables must have rows in the tail of the window ---
# Cheap 3-way UNION on the last 3 days. If any table is stale/empty, bail before
# spending 2+ hours on an extract that would silently produce empty parquets.
def _check_base_tables_have_rows(chk_start, chk_end):
    diag = f"""
    SELECT 'sales_order_items' AS source, count(*) AS row_count
    FROM dwh.fact_sales_order_item_details
    WHERE order_create_dt_ist >= DATE '{chk_start}'
      AND order_create_dt_ist <  DATE '{chk_end}' + INTERVAL '6' DAY
      AND cart_checkout_ts_ist >= DATE '{chk_start}'
      AND cart_checkout_ts_ist <  DATE '{chk_end}' + INTERVAL '1' DAY
      AND procured_quantity > 0 AND is_internal_order = FALSE
      AND order_current_status = 'DELIVERED'
    UNION ALL
    SELECT 'impressions', count(*) FROM category_etls.pid_impressions_and_atc
    WHERE at_date_ist BETWEEN DATE '{chk_start}' AND DATE '{chk_end}'
    UNION ALL
    SELECT 'keyword_atcs', count(*) FROM search_etls.keyword_city_product_additions
    WHERE at_date_ist >= DATE '{chk_start}'
      AND at_date_ist <  DATE '{chk_end}' + INTERVAL '1' DAY
    """
    df = sql(diag)
    print(df)
    empties = df[df['row_count'] == 0]['source'].tolist()
    assert not empties, f'Diag gate FAILED — empty tables: {empties}. Fix upstream before extracting.'
    print(f'  diag OK — all 3 base tables have rows in [{chk_start}, {chk_end}]')

_chk_start = (BACKFILL_END - timedelta(days=2)).isoformat()
_chk_end   = BACKFILL_END.isoformat()
print(f'\n--- Diag: tail of window {_chk_start} → {_chk_end} ---')
_check_base_tables_have_rows(_chk_start, _chk_end)

# --- Weekly batches — thread-parallel with per-source _DONE ------------------
# Three logical sources (bundling sales+city_totals since they share the same
# heavy fact table — running them together keeps Trino's query cache warm):
#   - sales_pair : sales.parquet + city_totals.parquet
#   - impressions : impressions_pid.parquet → impressions_brand.parquet
#   - keywords   : keywords.parquet
#
# Each source has its own _DONE_<source> marker per batch dir, so a partial
# batch (e.g. one source crashed mid-run) resumes cleanly.
# Batch is considered complete only when all three per-source _DONE files exist.
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

def weekly_batches(start, end):
    cur = start
    while cur <= end:
        wk_end = min(cur + timedelta(days=6), end)
        yield cur, wk_end
        cur = wk_end + timedelta(days=1)

# Each worker thread gets its own Trino connection. pencilbox connections are
# not thread-safe when shared, but creating one per thread is fine.
_thread_local = threading.local()
def _thread_conn():
    if not hasattr(_thread_local, 'conn'):
        _thread_local.conn = pb.get_connection("[Warehouse] Trino")
    return _thread_local.conn

def _run_to_parquet_chunked_conn(conn, sql_template, out_path, start, end, batch_size=50_000):
    """Thread-local variant of run_to_parquet_chunked — takes an explicit connection."""
    t0 = time.time()
    writer = None; schema = None; total = 0
    cur_day = date.fromisoformat(start); last_day = date.fromisoformat(end)
    while cur_day <= last_day:
        sql_day = sql_template.format(DAY=cur_day.isoformat())
        raw = conn.raw_connection(); cur = raw.cursor()
        td0 = time.time(); cur.execute(sql_day)
        cols = [d[0] for d in cur.description]
        day_rows = 0
        while True:
            batch = cur.fetchmany(batch_size)
            if not batch: break
            df = pd.DataFrame(batch, columns=cols)
            table = pa.Table.from_pandas(df, preserve_index=False)
            if writer is None:
                schema = table.schema
                writer = pq.ParquetWriter(out_path, schema, compression='snappy')
            else:
                table = table.cast(schema)
            writer.write_table(table)
            day_rows += len(batch)
        cur.close(); raw.close(); total += day_rows
        cur_day += timedelta(days=1)
    if writer is not None: writer.close()
    size_mb = os.path.getsize(out_path) / 1024**2 if os.path.exists(out_path) else 0
    print(f'  [{os.path.basename(out_path):>26s}] {total:>10,} rows, {size_mb:>6.1f} MB in {time.time()-t0:>5.0f}s', flush=True)
    return total

def _do_sales_pair(batch_dir, bs, be):
    done = os.path.join(batch_dir, '_DONE_sales_pair')
    if os.path.exists(done): return 'sales_pair: skip'
    conn = _thread_conn()
    _run_to_parquet_chunked_conn(conn, SALES_SQL_TEMPLATE,
                                 os.path.join(batch_dir, 'sales.parquet'),
                                 bs.isoformat(), be.isoformat())
    _run_to_parquet_chunked_conn(conn, CITY_TOTALS_SQL_TEMPLATE,
                                 os.path.join(batch_dir, 'city_totals.parquet'),
                                 bs.isoformat(), be.isoformat())
    open(done, 'w').close()
    return 'sales_pair: done'

def _do_impressions(batch_dir, bs, be):
    done = os.path.join(batch_dir, '_DONE_impressions')
    if os.path.exists(done): return 'impressions: skip'
    conn = _thread_conn()
    _run_to_parquet_chunked_conn(conn, IMPRESSIONS_SQL_TEMPLATE,
                                 os.path.join(batch_dir, 'impressions_pid.parquet'),
                                 bs.isoformat(), be.isoformat())
    # Local pid → brand rollup (uses products.parquet, single DuckDB in-mem)
    _con = duckdb.connect(':memory:')
    _con.execute(f"""
        COPY (
            SELECT i.day, i.city,
                   p.l0_category, p.l1_category, p.l2_category,
                   p.manufacturer, p.brand,
                   SUM(i.impressions) AS impressions,
                   SUM(i.atc)         AS atc,
                   SUM(i.atc_gmv)     AS atc_gmv
            FROM read_parquet('{batch_dir}/impressions_pid.parquet') i
            JOIN read_parquet('{products_path}')                    p ON p.product_id = i.pid
            GROUP BY 1,2,3,4,5,6,7
        ) TO '{batch_dir}/impressions_brand.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)
    _con.close()
    os.remove(os.path.join(batch_dir, 'impressions_pid.parquet'))
    open(done, 'w').close()
    return 'impressions: done'

def _do_keywords(batch_dir, bs, be):
    done = os.path.join(batch_dir, '_DONE_keywords')
    if os.path.exists(done): return 'keywords: skip'
    conn = _thread_conn()
    _run_to_parquet_chunked_conn(conn, KEYWORDS_SQL_TEMPLATE,
                                 os.path.join(batch_dir, 'keywords.parquet'),
                                 bs.isoformat(), be.isoformat())
    open(done, 'w').close()
    return 'keywords: done'

def _all_sources_done(batch_dir):
    return all(os.path.exists(os.path.join(batch_dir, f'_DONE_{s}'))
               for s in ('sales_pair', 'impressions', 'keywords'))

# --- Drive the parallel loop ------------------------------------------------
_done = 0
_skipped = 0
_batches = list(weekly_batches(BACKFILL_START, BACKFILL_END))
print(f'\nBackfill: {len(_batches)} weekly batches, 3-way parallel per batch.')

for bs, be in _batches:
    batch_dir = os.path.join(BACKFILL_DIR, f'{bs}_{be}')
    if _all_sources_done(batch_dir):
        print(f'  ✓ {bs}..{be}: skip (all sources done)')
        _skipped += 1
        continue
    os.makedirs(batch_dir, exist_ok=True)
    print(f'\n=== Batch {bs} → {be} ===')
    t_batch = time.time()
    with ThreadPoolExecutor(max_workers=3) as ex:
        futs = [ex.submit(fn, batch_dir, bs, be)
                for fn in (_do_sales_pair, _do_impressions, _do_keywords)]
        for f in as_completed(futs):
            print(f'  {f.result()}', flush=True)
    # Legacy compat: also emit the aggregate _DONE marker (older logic checks this)
    if _all_sources_done(batch_dir):
        open(os.path.join(batch_dir, '_DONE'), 'w').close()
    _done += 1
    print(f'  ✓ {bs}..{be} batch complete in {time.time()-t_batch:.0f}s')

print(f'\nbatches done now: {_done}, skipped (already): {_skipped}.')
print(f'Next step → run the merge cell (cell 14).')


## 6. Merge → partitioned Parquet + dim IDs (Option B)

**Single-tier**, full 90-day grain (day × city × brand). Achieves the
5 GB cap by 1) ZSTD compression, 2) replacing string dim columns with
INT IDs, 3) writing one Parquet file per day so the pod can hive-prune
by date.

Output:
- `/home/jovyan/final/dim/` — 5 small lookup tables
- `/home/jovyan/final/data/<source>/day=YYYY-MM-DD/data.parquet` — one
  file per day per source. Now includes `city_totals` for QC 2.1
  Cart/User Penetration.

`ingest_to_pod.sh` tars and ships the whole tree; the pod sets up
`read_parquet` views over the files.

In [ ]:
import duckdb, os, time, shutil

FINAL_DIR = '/home/jovyan/final'
DATA_DIR  = os.path.join(FINAL_DIR, 'data')
DIM_DIR   = os.path.join(FINAL_DIR, 'dim')

# Fresh build — clear targets
for d in (DATA_DIR, DIM_DIR):
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

con = duckdb.connect(':memory:')
t0 = time.time()

# ============================================================================
# DIM tables — small (~1 MB each), broadcast-joined at query time
# ============================================================================

con.execute(f"""
COPY (
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY manufacturer, brand, l0, l1, l2) AS INTEGER) AS brand_id,
        manufacturer, brand, l0, l1, l2
    FROM (
        SELECT DISTINCT manufacturer, brand, l0_cat AS l0, l1_cat AS l1, l2_cat AS l2
        FROM read_parquet('{BACKFILL_DIR}/*/sales.parquet')
        UNION
        SELECT DISTINCT manufacturer, brand, l0_category AS l0, l1_category AS l1, l2_category AS l2
        FROM read_parquet('{BACKFILL_DIR}/*/impressions_brand.parquet')
    )
) TO '{DIM_DIR}/dim_brand.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY city) AS INTEGER) AS city_id,
        city
    FROM (
        SELECT DISTINCT city FROM read_parquet('{BACKFILL_DIR}/*/sales.parquet')
        UNION
        SELECT DISTINCT city FROM read_parquet('{BACKFILL_DIR}/*/impressions_brand.parquet')
    )
) TO '{DIM_DIR}/dim_city.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY keyword) AS INTEGER) AS keyword_id,
        keyword
    FROM (SELECT DISTINCT keyword FROM read_parquet('{BACKFILL_DIR}/*/keywords.parquet'))
) TO '{DIM_DIR}/dim_keyword.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY COALESCE(ptype, '__none__')) AS INTEGER) AS ptype_id,
        ptype
    FROM (SELECT DISTINCT ptype FROM read_parquet('{BACKFILL_DIR}/*/keywords.parquet'))
) TO '{DIM_DIR}/dim_ptype.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY manufacturer, brand) AS INTEGER) AS brand_kw_id,
        manufacturer, brand
    FROM (
        SELECT DISTINCT manufacturer, brand
        FROM read_parquet('{BACKFILL_DIR}/*/keywords.parquet')
    )
) TO '{DIM_DIR}/dim_brand_kw.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(f'  dim tables written ({time.time()-t0:.0f}s)')

# ============================================================================
# FACT tables — ID-encoded, per-day partitioned, ZSTD compressed
# ============================================================================

# QC 2.1 — sales carries brand_orders (Cart Penetration numerator).
con.execute(f"""
COPY (
    SELECT
        s.day,
        CAST(c.city_id    AS INTEGER) AS city_id,
        CAST(b.brand_id   AS INTEGER) AS brand_id,
        s.qty, s.mrp, s.gmv,
        s.brand_orders
    FROM read_parquet('{BACKFILL_DIR}/*/sales.parquet') s
    JOIN read_parquet('{DIM_DIR}/dim_brand.parquet') b
        ON b.manufacturer = s.manufacturer AND b.brand = s.brand
        AND b.l0 = s.l0_cat AND b.l1 = s.l1_cat AND b.l2 = s.l2_cat
    JOIN read_parquet('{DIM_DIR}/dim_city.parquet') c ON c.city = s.city
) TO '{DATA_DIR}/sales' (
    FORMAT PARQUET, COMPRESSION ZSTD,
    PARTITION_BY (day), OVERWRITE_OR_IGNORE
)
""")
print(f'  sales partitioned ({time.time()-t0:.0f}s)')

# QC 2.1 — Cart Penetration denominator: total orders per (day, city, L2).
con.execute(f"""
COPY (
    SELECT
        ct.day,
        CAST(c.city_id AS INTEGER) AS city_id,
        ct.l0_cat, ct.l1_cat, ct.l2_cat,
        ct.city_orders
    FROM read_parquet('{BACKFILL_DIR}/*/city_totals.parquet') ct
    JOIN read_parquet('{DIM_DIR}/dim_city.parquet') c ON c.city = ct.city
) TO '{DATA_DIR}/city_totals' (
    FORMAT PARQUET, COMPRESSION ZSTD,
    PARTITION_BY (day), OVERWRITE_OR_IGNORE
)
""")
print(f'  city_totals partitioned ({time.time()-t0:.0f}s)')

con.execute(f"""
COPY (
    SELECT
        i.day,
        CAST(c.city_id    AS INTEGER) AS city_id,
        CAST(b.brand_id   AS INTEGER) AS brand_id,
        i.impressions, i.atc, i.atc_gmv
    FROM read_parquet('{BACKFILL_DIR}/*/impressions_brand.parquet') i
    JOIN read_parquet('{DIM_DIR}/dim_brand.parquet') b
        ON b.manufacturer = i.manufacturer AND b.brand = i.brand
        AND b.l0 = i.l0_category AND b.l1 = i.l1_category AND b.l2 = i.l2_category
    JOIN read_parquet('{DIM_DIR}/dim_city.parquet') c ON c.city = i.city
) TO '{DATA_DIR}/impressions' (
    FORMAT PARQUET, COMPRESSION ZSTD,
    PARTITION_BY (day), OVERWRITE_OR_IGNORE
)
""")
print(f'  impressions partitioned ({time.time()-t0:.0f}s)')

con.execute(f"""
COPY (
    SELECT
        k.day,
        CAST(kw.keyword_id  AS INTEGER) AS keyword_id,
        CAST(bk.brand_kw_id AS INTEGER) AS brand_kw_id,
        CAST(pt.ptype_id    AS INTEGER) AS ptype_id,
        k.searches, k.atcs, k.gmv
    FROM read_parquet('{BACKFILL_DIR}/*/keywords.parquet') k
    JOIN read_parquet('{DIM_DIR}/dim_keyword.parquet')  kw ON kw.keyword = k.keyword
    JOIN read_parquet('{DIM_DIR}/dim_brand_kw.parquet') bk
        ON bk.manufacturer = k.manufacturer AND bk.brand = k.brand
    JOIN read_parquet('{DIM_DIR}/dim_ptype.parquet')    pt
        ON pt.ptype IS NOT DISTINCT FROM k.ptype
) TO '{DATA_DIR}/keywords' (
    FORMAT PARQUET, COMPRESSION ZSTD,
    PARTITION_BY (day), OVERWRITE_OR_IGNORE
)
""")
print(f'  keywords partitioned ({time.time()-t0:.0f}s)')
con.close()

def _dirsize(p):
    total = 0
    for r, _, fs in os.walk(p):
        for f in fs:
            total += os.path.getsize(os.path.join(r, f))
    return total / 1024**2

print(f"\n{FINAL_DIR}:")
print(f"  dim/                {_dirsize(DIM_DIR):>8.1f} MB")
print(f"  data/sales/         {_dirsize(os.path.join(DATA_DIR, 'sales')):>8.1f} MB")
print(f"  data/city_totals/   {_dirsize(os.path.join(DATA_DIR, 'city_totals')):>8.1f} MB")
print(f"  data/impressions/   {_dirsize(os.path.join(DATA_DIR, 'impressions')):>8.1f} MB")
print(f"  data/keywords/      {_dirsize(os.path.join(DATA_DIR, 'keywords')):>8.1f} MB")
print(f"  TOTAL               {_dirsize(FINAL_DIR):>8.1f} MB  in {time.time()-t0:.0f}s")


## Done — sync `final/` to the pod

Output structure (Option B — partitioned + ID-encoded + ZSTD):
```
/home/jovyan/final/
  dim/ — dim_brand, dim_city, dim_keyword, dim_ptype, dim_brand_kw (~5 MB total)
  data/sales/day=YYYY-MM-DD/data.parquet         (one file per day, now with brand_orders)
  data/city_totals/day=YYYY-MM-DD/data.parquet   (new — Cart Penetration denominator)
  data/impressions/day=YYYY-MM-DD/data.parquet
  data/keywords/day=YYYY-MM-DD/data.parquet
```

Download the entire `final/` tree (or tar it locally), then:
```bash
./ingest_to_pod.sh ~/Downloads/final
```

**Pod-side follow-ups** when the new data lands:
- Rebuild `cube_sales_brand` to expose `brand_orders`
- Register a new `city_totals` view + a `cube_city_totals` cube
- Flip the Cart Penetration hero card from `pending` to live (sum brand_orders ÷ sum city_orders over the window)
- User Penetration card stays `pending` — no usable user-id column on the fact table.